# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# imports
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import os


In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
open_router_api_key = os.getenv('OPENROUTER_API_KEY')

openai = OpenAI()
open_router = "https://openrouter.ai/api/v1"
claude = OpenAI(base_url=open_router, api_key=open_router_api_key)

CHATGPT_MODEL = "gpt-5-mini"
CLAUDE_MODEL = "anthropic/claude-haiku-4.5"

MODEL_CONFIG = {
    "gpt-5-mini": {
        "client": openai,
        "model_name": CHATGPT_MODEL,
    },
    "claude-haiku-4.5": {
        "client": claude,
        "model_name": CLAUDE_MODEL,
    },
}

system_prompt = """You are a helpful coding tutor. Answer technical questions in a clear and concise manner. If the questions asks for an explanation of code, break down the code line by line and explain what each part does. If the question is about a concept, provide a clear and simple explanation with examples if possible. Always aim to make the answer easy to understand for someone who is learning to code. limit the response to 300 words."""


def get_code_explanation(message, history, model):
    config = MODEL_CONFIG.get(model)
    if not config:
        raise ValueError(f"Unsupported model: {model}")

    formatted_history = []
    if model == "claude-haiku-4.5":
        for idx, (user_msg, bot_msg) in enumerate(history):
            if idx == 0:
                user_msg = f"{system_prompt}\n{str(user_msg)}"
            formatted_history.append({"role": "user", "content": str(user_msg)})
            if bot_msg:
                formatted_history.append({"role": "assistant", "content": str(bot_msg)})
        if not history:
            message = f"{system_prompt}\n{str(message)}"
        messages = formatted_history + [{"role": "user", "content": str(message)}]
    else:
        for user_msg, bot_msg in history:
            formatted_history.append({"role": "user", "content": str(user_msg)})
            if bot_msg:
                formatted_history.append({"role": "assistant", "content": str(bot_msg)})
        messages = [{"role": "system", "content": system_prompt}] + formatted_history + [
            {"role": "user", "content": str(message)}]

    response_stream = config["client"].chat.completions.create(
        model=config["model_name"],
        messages=messages,
        stream=True
    )

    full_response = ""
    for chunk in response_stream:
        delta = getattr(chunk.choices[0].delta, "content", None)
        if delta:
            full_response += delta
            print(delta, end="", flush=True)
            yield full_response


with gr.Blocks() as ui:
    with gr.Row():
        model_dropdown = gr.Dropdown(
            label="Model",
            choices=["gpt-5-mini", "claude-haiku-4.5"],
            value="gpt-5-mini",
            interactive=True,
        )
    with gr.Row():
        chatbot = gr.ChatInterface(get_code_explanation, additional_inputs=[model_dropdown])

ui.launch()